# AG2 Multi-Agent RAG with LanceDB

<a href="https://colab.research.google.com/github/lancedb/vectordb-recipes/blob/main/examples/AG2-Multiagent-RAG-with-LanceDB/main.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook demonstrates how to use [AG2](https://ag2.ai/) (formerly AutoGen)
multi-agent conversations with [LanceDB](https://lancedb.github.io/lancedb/) for
Retrieval-Augmented Generation (RAG).

**LanceDB** is an open-source, embedded vector database built on Lance columnar
format. It runs in-process — no server needed — making it ideal for lightweight
RAG applications.

**AG2** is a multi-agent conversation framework with 500K+ monthly PyPI downloads,
4,300+ GitHub stars, and 400+ contributors.

Two AG2 agents collaborate:
- **Research Agent**: Retrieves relevant documents from LanceDB via semantic search
- **Analyst Agent**: Synthesizes retrieved information into comprehensive answers

In [ ]:
!pip install -q lancedb "ag2[openai]>=0.11.4,<1.0" sentence-transformers pandas

In [ ]:
import os

import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

from autogen import (
    AssistantAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
    UserProxyAgent,
)

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your key

## Create LanceDB Vector Database

LanceDB runs embedded — just point it at a local directory.
No Docker, no server process, no configuration needed.

We use `sentence-transformers` for embeddings so no API key is needed
for the vector search part.

In [ ]:
# Use sentence-transformers for embeddings (no API key needed)
embed_model = get_registry().get("sentence-transformers").create()


class Document(LanceModel):
    text: str = embed_model.SourceField()
    vector: Vector(embed_model.ndims()) = embed_model.VectorField()
    id: int


# Connect to embedded LanceDB
db = lancedb.connect("data/ag2-rag-lancedb")

# Sample documents about AI topics
documents = [
    {
        "text": "Retrieval-Augmented Generation (RAG) is a technique that combines "
        "information retrieval with language model generation. It first retrieves "
        "relevant documents from a knowledge base, then uses them as context for "
        "generating accurate, grounded responses.",
        "id": 0,
    },
    {
        "text": "Vector databases store data as high-dimensional vectors (embeddings) "
        "and enable fast similarity search. LanceDB is an embedded vector database "
        "built on the Lance columnar format, requiring no server — it runs "
        "in-process alongside your application.",
        "id": 1,
    },
    {
        "text": "Multi-agent systems use multiple AI agents that collaborate to solve "
        "complex tasks. Each agent can have specialized roles, tools, and knowledge. "
        "AG2 (formerly AutoGen) is a popular framework for building multi-agent "
        "conversations where agents use tools and coordinate through structured "
        "dialogue patterns.",
        "id": 2,
    },
    {
        "text": "Embedding models convert text into dense numerical vectors that "
        "capture semantic meaning. Similar texts produce similar vectors, enabling "
        "semantic search. Sentence-transformers provide high-quality embeddings "
        "that can run locally without an API key.",
        "id": 3,
    },
    {
        "text": "Chunking is the process of splitting documents into smaller pieces "
        "for embedding and retrieval. Common strategies include fixed-size chunking, "
        "recursive character splitting, and semantic chunking. Proper chunking "
        "improves retrieval quality significantly.",
        "id": 4,
    },
]

# Create table — LanceDB auto-computes embeddings via the model
table = db.create_table("documents", schema=Document, data=documents, mode="overwrite")
print(f"Indexed {table.count_rows()} documents into LanceDB")

## Test Semantic Search

Verify that LanceDB retrieval works before connecting it to AG2 agents.

In [ ]:
# Semantic search — LanceDB handles embedding the query automatically
results = table.search("What is RAG?").limit(3).to_pandas()

for _, row in results.iterrows():
    print(f"Score: {row['_distance']:.4f}")
    print(f"  {row['text'][:100]}...")
    print()

## Define LanceDB Search Tool and AG2 Agents

Create a search function that AG2 agents will use as a registered tool
to retrieve relevant documents from LanceDB.

In [ ]:
# Configure AG2
llm_config = LLMConfig(
    {
        "model": "gpt-4o-mini",
        "api_key": os.getenv("OPENAI_API_KEY"),
        "api_type": "openai",
    }
)


def is_termination_msg(msg):
    content = msg.get("content", "") or ""
    return "TERMINATE" in content


# Create UserProxy (orchestrator)
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=is_termination_msg,
    code_execution_config=False,
)

# Research Agent — retrieves documents from LanceDB
research_agent = AssistantAgent(
    name="research_agent",
    system_message=(
        "You are a Research Agent. Your role is to retrieve relevant documents "
        "from the LanceDB vector database using the search tool. Always use the "
        "search_documents tool to find information before answering. Present the "
        "retrieved documents clearly with their relevance scores."
    ),
    llm_config=llm_config,
)

# Analyst Agent — synthesizes information
analyst_agent = AssistantAgent(
    name="analyst_agent",
    system_message=(
        "You are an Analyst Agent. Your role is to synthesize information "
        "retrieved by the Research Agent into comprehensive, well-structured "
        "answers. Wait for the Research Agent to provide retrieved documents, "
        "then analyze and synthesize the findings. When done, say TERMINATE."
    ),
    llm_config=llm_config,
)


# Register search tool using AG2 decorator pattern
@user_proxy.register_for_execution()
@research_agent.register_for_llm(
    description="Search the LanceDB vector database for relevant documents"
)
def search_documents(query: str, num_results: int = 3) -> str:
    """Search LanceDB for documents matching the query."""
    results = table.search(query).limit(num_results).to_pandas()
    output = []
    for _, row in results.iterrows():
        output.append(f"[Score: {row['_distance']:.4f}] {row['text']}")
    return "\n\n".join(output) if output else "No relevant documents found."

## Run Multi-Agent RAG Conversation

Create a GroupChat where the Research Agent and Analyst Agent
collaborate to answer questions using LanceDB-backed retrieval.

In [ ]:
# Create GroupChat with both agents
group_chat = GroupChat(
    agents=[user_proxy, research_agent, analyst_agent],
    messages=[],
    max_round=10,
)

manager = GroupChatManager(groupchat=group_chat, llm_config=llm_config)

# Run multi-agent RAG conversation
result = user_proxy.run(
    manager,
    message="What is RAG and how do multi-agent systems enhance it? "
    "Search the knowledge base for relevant information.",
)
result.process()

## Summary

This notebook demonstrated:

1. **LanceDB as embedded vector database** — no server needed, just `lancedb.connect()`
2. **AG2 multi-agent orchestration** — Research Agent retrieves, Analyst Agent synthesizes
3. **Tool registration** — AG2's decorator pattern wraps LanceDB search as an agent tool
4. **GroupChat coordination** — agents collaborate through structured conversation

### Learn More
- [AG2 Documentation](https://docs.ag2.ai/)
- [LanceDB Documentation](https://lancedb.github.io/lancedb/)
- [AG2 GitHub](https://github.com/ag2ai/ag2)
- [LanceDB GitHub](https://github.com/lancedb/lancedb)